# Learning to Count Everything with Falcon Perception

This notebook demonstrates a **"Count Everything"** application built on top of **Falcon Perception**.

**How it works:**
1. A user submits an **image** containing objects.
2. The user asks a natural-language question, e.g. *"How many bears are there in the image?"*
3. The system **parses** the question to extract the target object name (e.g. `bear`).
4. Falcon Perception runs **open-vocabulary detection** to locate every instance.
5. The system **counts** the detections and returns the answer with a visual overlay.

Because Falcon Perception is open-vocabulary, this works for **any** object type and no retraining needed.

> **Key design choice:** We use `detection` mode (bounding boxes only) rather than `segmentation` mode. This skips the expensive HR upsampler, giving us faster inference. Counting only needs to know *where* objects are, not their pixel-precise masks.

## Prerequisites

- A CUDA-capable GPU (tested on RTX 4000 Ada 20 GB with `Falcon-Perception` + bfloat16)
- `falcon-perception` package

In [ ]:
%pip install falcon-perception
%pip install Pillow

In [ ]:
import torch
from falcon_perception import (
    PERCEPTION_MODEL_ID,
    build_prompt_for_task,
    cuda_timed,
    load_and_prepare_model,
    setup_torch_config,
)
from falcon_perception.data import ImageProcessor, load_image, stream_samples_from_hf_dataset
from falcon_perception.paged_inference import (
    PagedInferenceEngine,
    SamplingParams,
    Sequence,
)
from falcon_perception.visualization_utils import (
    overlay_detections_on_image_v2,
    pair_bbox_entries,
)

setup_torch_config()

## 1. Load Model & Build Engine

We load Falcon Perception with the same configuration as the main notebook.
Detection mode does not require the HR upsampler, but we still enable `enable_hr_cache` in case users want to experiment with segmentation-based counting later.

In [ ]:
MAX_SEQ_LENGTH = 8192
MIN_IMAGE_SIZE = 256
MAX_IMAGE_SIZE = 1024

model, tokenizer, model_args = load_and_prepare_model(
    hf_model_id=PERCEPTION_MODEL_ID,
    dtype="bfloat16",
    compile=True,
)

image_processor = ImageProcessor(patch_size=16, merge_size=1)
stop_token_ids = [tokenizer.eos_token_id, tokenizer.end_of_query_token_id]

# Manual engine config sized for ~5 GB free VRAM (300M model + bfloat16)
cfg = dict(
    n_pages=128,
    page_size=128,
    max_batch_size=4,
    prefill_length_limit=8192,
    max_hr_cache_entries=0,
    max_image_size=MAX_IMAGE_SIZE,
)

engine = PagedInferenceEngine(
    model, tokenizer, image_processor,
    max_seq_length=MAX_SEQ_LENGTH,
    enable_hr_cache=False,
    capture_cudagraph=True,
    **cfg,
)

### Warmup

Absorbs `torch.compile` JIT and CUDA graph capture costs.

In [ ]:
warmup_sample = stream_samples_from_hf_dataset("tiiuae/PBench", split="level_1", limit=1)[0]
warmup_prompt = build_prompt_for_task(str(warmup_sample["expression"]), "detection")
warmup_seq = Sequence(
    text=warmup_prompt,
    image=warmup_sample["image"],
    min_image_size=MIN_IMAGE_SIZE,
    max_image_size=MAX_IMAGE_SIZE,
    task="detection",
)
sp = SamplingParams(stop_token_ids=stop_token_ids)

print("Warmup run (detection mode) ...")
engine.generate([warmup_seq], sampling_params=sp, use_tqdm=False, print_stats=False)
print("Warmup done")

---

## 2. Question Parsing

We need to extract the **target object** from a natural-language counting question.

Examples:
- *"How many bears are there in the image?"* → `bear`
- *"Count the red cars"* → `red car`
- *"How many people?"* → `person`

The parser handles several question patterns and applies basic singularisation (e.g. `bears` → `bear`, `buses` → `bus`).

In [ ]:
import re

IRREGULAR_PLURALS = {
    "people": "person", "men": "man", "women": "woman",
    "children": "child", "mice": "mouse", "geese": "goose",
    "teeth": "tooth", "feet": "foot", "oxen": "ox",
    "sheep": "sheep", "fish": "fish", "deer": "deer",
}

QUESTION_PATTERNS = [
    re.compile(r"how\s+many\s+(.+?)\s+(?:are|is)\s+(?:there\s+)?(?:in|on|at)", re.IGNORECASE),
    re.compile(r"how\s+many\s+(.+?)\s+(?:are|is)\s+there", re.IGNORECASE),
    re.compile(r"how\s+many\s+(.+?)\s+(?:can|do)\s+you\s+(?:see|count|find|spot)", re.IGNORECASE),
    re.compile(r"how\s+many\s+(.+?)[\?\s]*$", re.IGNORECASE),
    re.compile(r"count\s+(?:the\s+|all\s+(?:the\s+)?)?(.+?)[\?\.\s]*$", re.IGNORECASE),
    re.compile(r"(?:number|amount)\s+of\s+(.+?)[\?\.\s]*$", re.IGNORECASE),
]


def singularise(word: str) -> str:
    word = word.strip().lower()
    if word in IRREGULAR_PLURALS:
        return IRREGULAR_PLURALS[word]
    if word.endswith("ies") and len(word) > 4:
        return word[:-3] + "y"
    if word.endswith(("ses", "xes", "zes", "ches", "shes")):
        return word[:-2]
    if word.endswith("s") and not word.endswith("ss"):
        return word[:-1]
    return word


def extract_object_from_question(question: str) -> str:
    question = question.strip()
    for pattern in QUESTION_PATTERNS:
        match = pattern.search(question)
        if match:
            raw = match.group(1).strip().rstrip("?.! ")
            words = raw.split()
            words[-1] = singularise(words[-1])
            return " ".join(words)
    return question.rstrip("?.! ").strip()


# Quick tests
test_cases = [
    "How many bears are there in the image?",
    "How many red cars are there?",
    "How many people can you see?",
    "count the yellow boxes",
    "How many sheep?",
    "dog",
]
print("Question parsing tests:")
for q in test_cases:
    print(f"  {q!r:55s} → {extract_object_from_question(q)!r}")

---

## 3. Core Counting Engine

The `count_objects` function ties everything together:
1. Parse the question → extract object name
2. Build a detection prompt via `build_prompt_for_task(object_name, "detection")`
3. Run inference with `engine.generate()`
4. Count bounding boxes via `pair_bbox_entries()`
5. Return the count and optionally a visual overlay

We also provide `count_and_display` for a user-friendly output with the annotated image.

In [ ]:
def count_objects(image, question, coord_dedup_threshold=0.01):
    object_name = extract_object_from_question(question)
    prompt = build_prompt_for_task(object_name, "detection")
    seq = Sequence(
        text=prompt, image=image,
        min_image_size=MIN_IMAGE_SIZE,
        max_image_size=MAX_IMAGE_SIZE,
        task="detection",
    )
    sp = SamplingParams(
        stop_token_ids=stop_token_ids,
        coord_dedup_threshold=coord_dedup_threshold,
    )

    with cuda_timed() as t:
        engine.generate([seq], sampling_params=sp, use_tqdm=False, print_stats=False)

    bboxes = pair_bbox_entries(seq.output_aux.bboxes_raw)
    return {
        "object": object_name,
        "count": len(bboxes),
        "bboxes": bboxes,
        "sequence": seq,
        "elapsed_s": t.elapsed,
    }

def count_and_display(image, question, coord_dedup_threshold=0.01, max_display_side=1024):
    result = count_objects(image, question, coord_dedup_threshold)
    obj, count = result["object"], result["count"]

    if count == 0:
        answer = f"I don't see any {obj} in the image."
    elif count == 1:
        answer = f"There is 1 {obj} in the image."
    else:
        answer = f"There are {count} {obj}s in the image."

    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"   (detected {count} instance(s) in {result['elapsed_s']:.2f}s)")

    # Render overlay with bounding boxes
    if result["bboxes"]:
        dets = [
            {"xy": {"x": b["x"], "y": b["y"]}, "hw": {"w": b["w"], "h": b["h"]}}
            for b in result["bboxes"]
        ]
        overlay = overlay_detections_on_image_v2(
            image, dets, draw_bbox=True, masks_are_binary=True,
        )
        display(Image.fromarray(overlay))

    return result

---

## 4. Basic Counting Demo

Let's start with a simple example. We load an image and ask a straightforward counting question.

In [ ]:
demo_img = load_image(
    "https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/pexels-tahaasamett-10540813.jpg"
)
print(f"Demo image size: {demo_img.size}")
display(cap_display(demo_img))

In [ ]:
# Simple counting question
count_and_display(demo_img, "How many boxes are there in the image?")

In [ ]:
# Counting with an attribute — only count a specific colour
count_and_display(demo_img, "How many purple boxes are there?")

---

## 5. Multi-Object Counting: Ask Multiple Questions on One Image

A common use case: scan an image for several object types and summarise the counts.
Since we're querying the **same image** repeatedly, the HR feature cache (even though we use detection mode) helps the engine optimise.

In [ ]:
def count_multiple_objects(image: Image.Image, questions: list[str], coord_dedup_threshold: float = 0.01):
    """Run multiple counting questions on the same image and print a summary table."""
    results = []
    total_time = 0.0

    for question in questions:
        result = count_objects(image, question, coord_dedup_threshold)
        results.append(result)
        total_time += result["elapsed_s"]

    # Print summary table
    print(f"{'Object':<25s} {'Count':>6s} {'Time':>8s}")
    print("-" * 42)
    for r in results:
        print(f"{r['object']:<25s} {r['count']:>6d} {r['elapsed_s']:>7.2f}s")
    print("-" * 42)
    print(f"{'Total':.<25s} {sum(r['count'] for r in results):>6d} {total_time:>7.2f}s")

    return results


# Ask multiple questions about the demo image
questions = [
    "How many boxes are there?",
    "How many plants can you see?",
    "How many bottles are there in the image?",
]
multi_results = count_multiple_objects(demo_img, questions)

---

## 6. Dense Counting: Many Small Objects

Counting becomes challenging when images contain many small, tightly packed objects (e.g. a parking lot full of cars, a flock of sheep).

The `coord_dedup_threshold` parameter is critical here:
- **`0`** —> keep all predictions, including near-duplicate overlapping boxes → over-counts.
- **`0.01`** —> merge boxes whose centres are within 1% of normalised distance → cleaner counts.

We compare both settings on the same dense scene.

In [ ]:
dense_img = load_image(
    "https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/sheep.jpg"
)

In [ ]:
# Without dedup
result_no_dedup = count_and_display(dense_img, "How many sheep are there?", coord_dedup_threshold=0)

# With dedup
result_dedup = count_and_display(dense_img, "How many sheep are there?", coord_dedup_threshold=0.01)

print(f"\nDedup reduced count from {result_no_dedup['count']} → {result_dedup['count']} "
      f"(removed {result_no_dedup['count'] - result_dedup['count']} duplicate detections)")

> **Hardware note:** Dense counting on closely packed objects is particularly sensitive to numerical precision. For best results in this scenario, we recommend loading the model in **float32** (`dtype="float32"`) rather than bfloat16. This requires approximately 2× the VRAM (~20 GB) but produces significantly more accurate detection boundaries, reducing both missed detections and false duplicates. The code above is ready to run as-is. You simply update the `dtype` parameter in `load_and_prepare_model()` when sufficient GPU memory is available.

> ⚠️ **Note:** A `coord_dedup_threshold` of `0.01` is a good default for dense scenes because it helps reduce over-counting from overlapping detections.

---

## 7. Different Object Types

Falcon Perception is **open-vocabulary** which can count anything it can detect.
Let's demonstrate on a different image with a different object type.

In [ ]:
nyt_img = load_image(
    "https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/nyt.jpg"
)
print(f"Image size: {nyt_img.size}")

# Count different object types in the same image
count_and_display(nyt_img, "How many people are there in the image?")

In [ ]:
count_and_display(nyt_img, "Count the photographs")

---

## 8. Try It Yourself: Custom Image & Question

Load your own image (from a URL or local path) and ask any counting question.

**Usage:**
- Change `YOUR_IMAGE` to a URL or local file path
- Change `YOUR_QUESTION` to any counting question (or just type an object name)

In [ ]:
# ── Edit these two variables ──────────────────────────────────────
YOUR_IMAGE = "https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/seagull.jpg"
YOUR_QUESTION = "How many black cormorants are there?"
# ─────────────────────────────────────────────────────────────────

user_img = load_image(YOUR_IMAGE)
count_and_display(user_img, YOUR_QUESTION)

---

## 9. Batch Object Inventory

Given a list of object categories, scan an image and produce an inventory — useful for warehouse inspection, wildlife surveys, or retail shelf auditing.

In [ ]:
def object_inventory(
    image: Image.Image,
    object_names: list[str],
    coord_dedup_threshold: float = 0.01,
) -> dict[str, int]:
    """
    Count each object type in the image and return {object_name: count}.
    Only includes objects with count > 0.
    """
    inventory = {}
    total_time = 0.0

    for name in object_names:
        result = count_objects(image, name, coord_dedup_threshold)
        inventory[result["object"]] = result["count"]
        total_time += result["elapsed_s"]

    # Display results
    print(f"{'Object':<25s} {'Count':>6s}")
    print("=" * 33)
    for obj, cnt in sorted(inventory.items(), key=lambda x: -x[1]):
        marker = "  ✓" if cnt > 0 else ""
        print(f"{obj:<25s} {cnt:>6d}{marker}")
    print("=" * 33)
    found = sum(1 for c in inventory.values() if c > 0)
    print(f"Found {found}/{len(object_names)} object types | Total: {sum(inventory.values())} instances | {total_time:.2f}s")

    return inventory


# Run an inventory scan
inventory = object_inventory(demo_img, [
    "box", "plant", "bottle", "book", "leaf", "boat",
])

---

## 10. Optional: Counting with Segmentation Mode

Throughout this notebook we used **detection mode** (bounding boxes only) because counting only needs to know *where* objects are — not their pixel-precise boundaries. This keeps inference fast by skipping the HR upsampler.

However, if you need **mask-level detail** (e.g. measuring object area, pixel-level editing, or visual verification of boundaries), Falcon Perception also supports **segmentation mode**. Here's how to adapt the workflow:

1. **Rebuild the engine** with the HR cache enabled:
   - Set `enable_hr_cache=True` and `max_hr_cache_entries=4` (or higher depending on your batch size).
2. **Change the task** in `build_prompt_for_task()` and `Sequence`:
   - Use `"segmentation"` instead of `"detection"`.
3. **Access masks** from the output:
   - After generation, `seq.output_aux.masks` contains per-instance binary masks.
   - Pass them to `overlay_detections_on_image_v2(..., masks=masks)` for visualisation.

> **Trade-off:** Segmentation mode is noticeably slower due to the HR upsampler. For pure counting tasks, detection mode is recommended. Use segmentation only when you actually need the mask output.

---

## Summary

This notebook demonstrated a **"Learning to Count Everything"** application built on Falcon Perception:

| Section | What it covers |
|---------|---------------|
| **Question Parsing** | Extract the target object from natural-language questions (regex-based, handles plurals) |
| **Core Counting** | `count_objects()` — parse question → detect → count bounding boxes |
| **Basic Demo** | Single-object counting with and without attribute filtering |
| **Multi-Object** | Ask multiple counting questions on one image |
| **Dense Counting** | `coord_dedup_threshold` for accurate counts in crowded scenes |
| **Different Objects** | Open-vocabulary counting works for any object type |
| **Try It Yourself** | Plug in your own image + question |
| **Batch Inventory** | Scan an image for a list of object categories |
| **Segmentation Mode** | Optional pixel-precise masks alongside counts |

**Key takeaways:**
- **Detection mode** is preferred for counting — faster since it skips the HR upsampler.
- **`coord_dedup_threshold=0.01`** is recommended for dense scenes to avoid over-counting.
- The system is **open-vocabulary** — no retraining needed for new object types.
- For pixel-precise output, switch to **segmentation mode** at the cost of higher latency.